# Periodic committor comparison

This repository is private. The first code cell therefore asks once for a GitHub personal-access token with repository read access. Paste the token when prompted; the input is hidden, and the notebook does not store it in the cloned repository. 

Before running, choose **Runtime → Change runtime type → T4 GPU → Save**. Then run the cells from top to bottom. The training cell saves checkpoints and figures in Google Drive.

In [ ]:
from pathlib import Path
from getpass import getpass
import base64
import shutil
import subprocess

from google.colab import drive
drive.mount('/content/drive')

REPO = Path('/content/boundary-adapted-pinns')
REPO_URL = 'https://github.com/n2pakbong/boundary-adapted-pinns.git'
BRANCH = 'rebuild/committor-ready-core'

if not (REPO / 'pyproject.toml').is_file():
    if REPO.exists():
        # This fixed /content path can only contain an incomplete clone.
        shutil.rmtree(REPO)
    token = getpass('Paste your GitHub token, then press Enter: ')
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    try:
        subprocess.run(
            ['git', '-c', f'http.extraheader=AUTHORIZATION: basic {auth}',
             'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(REPO)],
            check=True,
        )
    finally:
        del token, auth
    # Keep the normal token-free URL for later Git operations.
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', REPO_URL], check=True)
else:
    print('Using the existing clone at', REPO)

%cd /content/boundary-adapted-pinns
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.[dev]'], check=True)
subprocess.run(['python', '-m', 'pytest', '-q'], check=True)
assert shutil.which('met-pinn-periodic-comparison'), 'The experiment command was not installed.'
subprocess.run(['nvidia-smi'], check=True)
print('Setup passed: the experiment command is installed.')

In [ ]:
from datetime import datetime
from pathlib import Path
import subprocess

run_name = 'overnight_' + datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_ROOT = f'/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/{run_name}'
Path(RUN_ROOT).mkdir(parents=True, exist_ok=False)
print('Results will be saved in:', RUN_ROOT)

subprocess.run([
    'met-pinn-periodic-comparison',
    'configs/periodic_committor/overnight_comparison.json',
    '--output-dir', RUN_ROOT,
], check=True)

In [ ]:
import json
from IPython.display import Image, display

with open(f'{RUN_ROOT}/summary.json') as handle:
    print(json.dumps(json.load(handle), indent=2))
for filename in ['losses.png', 'committor_boundary_adapted.png', 'committor_naive_product.png', 'committor_penalty.png']:
    display(Image(filename=f'{RUN_ROOT}/{filename}'))

## If Colab disconnects

The Drive folder printed by the training cell contains `last.pt` checkpoints. In a fresh runtime, rerun the first setup cell, set `RUN_ROOT` below to the interrupted Drive directory, and run the resume cell.

In [ ]:
# Replace the text between quotes with the interrupted Drive directory.
# RUN_ROOT = '/content/drive/MyDrive/boundary-adapted-pinns-runs/periodic_committor/overnight_YYYYMMDD_HHMMSS'
# subprocess.run([
#     'met-pinn-periodic-comparison',
#     'configs/periodic_committor/overnight_comparison.json',
#     '--output-dir', RUN_ROOT,
#     '--resume',
# ], check=True)